# Router 2 Experiment

This notebook is for testing a different router style without copying the full router pipeline again. It reuses the existing chronological data splits, frozen expert loading, scaling, and error helpers from `scripts/chronological_expert_training.py`, then swaps in a new router architecture.

## What Stays The Same

- The selected experts come from `scripts/router_model_config.py`.
- Selected expert checkpoints must already exist under `checkpoints/candidates/`.
- The scaler is fit only on `expert_train`.
- Router 2 trains only on `router_train` and selects the best checkpoint on `router_val`.
- Final evaluation uses the untouched `test` split once.

Run Stages 1 and 2 in `router.ipynb` first if you do not already have the expert checkpoints.

In [1]:
from pathlib import Path
import csv
import json
import sys
from typing import Optional, Sequence, Tuple, Union

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.chronological_expert_training import (
    DEFAULT_INPUT_LEN,
    DEFAULT_NUM_FEATURES,
    DEFAULT_OUTPUT_LEN,
    _accumulate_errors,
    _assert_full_data_contract,
    _assert_no_expert_gradients,
    _check_shapes,
    _call_expert_model,
    _dataset_config_summary,
    _load_torch_checkpoint,
    _prepare_forecasting_batch,
    assert_experts_frozen,
    build_selected_candidate_experts,
    load_full_chronological_data,
    prepare_chronological_dataloaders,
)
from basicts.scaler import ZScoreScaler

torch.set_float32_matmul_precision("high") if hasattr(torch, "set_float32_matmul_precision") else None

## Router 2 Style

This router is intentionally different from `PredictionAwareRouter`. Instead of a CNN history encoder, it summarizes the input history with simple statistics: mean, standard deviation, last value, and change from first to last. It combines those history features with DLinear's forecast, iTransformer's forecast, their absolute disagreement, and their signed difference.

In [2]:
class Router2FeatureRouter(nn.Module):
    """Alternative per-step router using history statistics and selected expert forecasts."""

    router_type = "original"
    display_name = "Router 2 feature router"
    expert_prediction_layout = "expert_feature"

    def __init__(
        self,
        input_len: int = DEFAULT_INPUT_LEN,
        forecast_horizon: int = DEFAULT_OUTPUT_LEN,
        num_features: int = DEFAULT_NUM_FEATURES,
        num_experts: int = 2,
        history_hidden_size: int = 64,
        routing_hidden_size: int = 96,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.input_len = input_len
        self.forecast_horizon = forecast_horizon
        self.num_features = num_features
        if num_experts < 2:
            raise ValueError("Router 2 requires at least two selected experts")
        self.num_experts = num_experts
        self.history_hidden_size = history_hidden_size
        self.routing_hidden_size = routing_hidden_size
        self.dropout = dropout

        self.history_encoder = nn.Sequential(
            nn.Linear(4 * num_features, history_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        per_step_input_size = history_hidden_size + (num_experts + 2) * num_features
        self.routing_head = nn.Sequential(
            nn.Linear(per_step_input_size, routing_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(routing_hidden_size, num_experts),
        )

    def config_dict(self) -> dict:
        return {
            "input_len": self.input_len,
            "forecast_horizon": self.forecast_horizon,
            "num_features": self.num_features,
            "num_experts": self.num_experts,
            "history_hidden_size": self.history_hidden_size,
            "routing_hidden_size": self.routing_hidden_size,
            "dropout": self.dropout,
        }

    def _history_summary(self, historical_input: torch.Tensor) -> torch.Tensor:
        mean = historical_input.mean(dim=1)
        std = historical_input.std(dim=1, unbiased=False)
        last = historical_input[:, -1, :]
        change = historical_input[:, -1, :] - historical_input[:, 0, :]
        return torch.cat((mean, std, last, change), dim=-1)

    def forward(
        self,
        historical_input: torch.Tensor,
        expert_predictions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch_size = historical_input.shape[0]
        expected_history = (batch_size, self.input_len, self.num_features)
        expected_stack = (batch_size, self.forecast_horizon, self.num_experts, self.num_features)
        if tuple(historical_input.shape) != expected_history:
            raise ValueError(f"historical_input shape {tuple(historical_input.shape)} does not match {expected_history}")
        if tuple(expert_predictions.shape) != expected_stack:
            raise ValueError(f"expert_predictions shape {tuple(expert_predictions.shape)} does not match {expected_stack}")

        history_context = self.history_encoder(self._history_summary(historical_input))
        history_context = history_context.unsqueeze(1).expand(-1, self.forecast_horizon, -1)
        flattened_predictions = expert_predictions.reshape(
            batch_size,
            self.forecast_horizon,
            self.num_experts * self.num_features,
        )
        mean_prediction = expert_predictions.mean(dim=2)
        disagreement = torch.mean(torch.abs(expert_predictions - mean_prediction.unsqueeze(2)), dim=2)
        router_features = torch.cat(
            (history_context, flattened_predictions, mean_prediction, disagreement),
            dim=-1,
        )
        router_scores = self.routing_head(router_features)
        router_weights = torch.softmax(router_scores, dim=-1)
        mixed_prediction = torch.sum(router_weights.unsqueeze(-1) * expert_predictions, dim=2)
        if not torch.allclose(router_weights.sum(dim=-1), torch.ones_like(router_weights[..., 0]), atol=1e-6):
            raise AssertionError("Router 2 weights do not sum to 1")
        return mixed_prediction, router_weights, router_scores


class Router2TemporalResidualBlock(nn.Module):
    """Length-preserving temporal convolution block for [B, T, C] features."""

    def __init__(self, embedding_dim: int, kernel_size: int, dilation: int, dropout: float) -> None:
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=embedding_dim,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
        )
        self.norm = nn.LayerNorm(embedding_dim)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        residual = sequence
        encoded = self.conv(sequence.transpose(1, 2)).transpose(1, 2)
        if encoded.shape[1] != residual.shape[1]:
            encoded = encoded[:, : residual.shape[1], :]
        encoded = self.norm(encoded)
        encoded = self.activation(encoded)
        encoded = self.dropout(encoded)
        return residual + encoded


class MultiscaleTCNExpertEmbeddingRouter(nn.Module):
    """Soft prediction-aware TCN router with RouterDC-style trainable expert embeddings."""

    router_type = "multiscale_tcn_expert_embeddings"
    display_name = "Multiscale TCN expert-embedding router"
    expert_prediction_layout = "feature_expert"

    def __init__(
        self,
        input_len: int = DEFAULT_INPUT_LEN,
        forecast_horizon: int = DEFAULT_OUTPUT_LEN,
        num_features: int = DEFAULT_NUM_FEATURES,
        num_experts: int = 2,
        embedding_dim: int = 64,
        kernel_size: int = 5,
        dilations: Sequence[int] = (1, 2, 4, 8, 16),
        dropout: float = 0.1,
        num_heads: int = 4,
        prediction_hidden_size: int = 64,
    ) -> None:
        super().__init__()
        self.input_len = input_len
        self.forecast_horizon = forecast_horizon
        self.num_features = num_features
        if num_experts < 2:
            raise ValueError("Router 2 requires at least two selected experts")
        self.num_experts = num_experts
        self.embedding_dim = embedding_dim
        self.kernel_size = kernel_size
        self.dilations = tuple(dilations)
        self.dropout = dropout
        self.num_heads = num_heads
        self.prediction_hidden_size = prediction_hidden_size

        self.input_projection = nn.Linear(num_features, embedding_dim)
        self.history_encoder = nn.Sequential(
            *[
                Router2TemporalResidualBlock(
                    embedding_dim=embedding_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
                for dilation in self.dilations
            ]
        )
        self.horizon_queries = nn.Parameter(torch.randn(forecast_horizon, embedding_dim) * 0.02)
        self.horizon_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.variable_embeddings = nn.Parameter(torch.randn(num_features, embedding_dim) * 0.02)
        self.prediction_encoder = nn.Sequential(
            nn.Linear(3 * num_experts, prediction_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(prediction_hidden_size, embedding_dim),
            nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, embedding_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.expert_embeddings = nn.Parameter(
            torch.randn(num_experts, embedding_dim)
        )
        nn.init.normal_(self.expert_embeddings, mean=0.0, std=0.02)

    def config_dict(self) -> dict:
        return {
            "router_type": self.router_type,
            "input_len": self.input_len,
            "forecast_horizon": self.forecast_horizon,
            "num_features": self.num_features,
            "num_experts": self.num_experts,
            "embedding_dim": self.embedding_dim,
            "kernel_size": self.kernel_size,
            "dilations": list(self.dilations),
            "dropout": self.dropout,
            "num_heads": self.num_heads,
            "prediction_hidden_size": self.prediction_hidden_size,
        }

    def forward(
        self,
        historical_input: torch.Tensor,
        expert_predictions: torch.Tensor,
        return_intermediates: bool = False,
    ):
        batch_size = historical_input.shape[0]
        num_experts = self.num_experts
        assert historical_input.shape[1:] == (self.input_len, self.num_features)
        assert expert_predictions.shape[1:] == (
            self.forecast_horizon,
            self.num_features,
            num_experts,
        )

        projected_history = self.input_projection(historical_input)
        encoded_history = self.history_encoder(projected_history)
        horizon_queries = self.horizon_queries.unsqueeze(0).expand(batch_size, -1, -1)
        horizon_history, _ = self.horizon_attention(
            query=horizon_queries,
            key=encoded_history,
            value=encoded_history,
            need_weights=False,
        )
        horizon_variable_features = (
            horizon_history.unsqueeze(2) + self.variable_embeddings.unsqueeze(0).unsqueeze(0)
        )

        mean_prediction = expert_predictions.mean(dim=-1, keepdim=True)
        signed_disagreement = expert_predictions - mean_prediction
        prediction_features = torch.cat(
            (expert_predictions, signed_disagreement, signed_disagreement.abs()),
            dim=-1,
        )
        prediction_embedding = self.prediction_encoder(prediction_features)
        fused_features = self.fusion(horizon_variable_features + prediction_embedding)

        router_features = F.normalize(fused_features, p=2, dim=-1)
        expert_vectors = F.normalize(self.expert_embeddings, p=2, dim=-1)
        logits = torch.einsum(
            "btvd,md->btvm",
            router_features,
            expert_vectors
        )
        weights = torch.softmax(logits, dim=-1)
        mixed_prediction = (
            weights * expert_predictions
        ).sum(dim=-1)

        assert projected_history.shape == (batch_size, self.input_len, self.embedding_dim)
        assert encoded_history.shape == (batch_size, self.input_len, self.embedding_dim)
        assert horizon_history.shape == (batch_size, self.forecast_horizon, self.embedding_dim)
        assert horizon_variable_features.shape == (
            batch_size,
            self.forecast_horizon,
            self.num_features,
            self.embedding_dim,
        )
        assert prediction_embedding.shape == (
            batch_size,
            self.forecast_horizon,
            self.num_features,
            self.embedding_dim,
        )
        assert fused_features.shape == (
            batch_size,
            self.forecast_horizon,
            self.num_features,
            self.embedding_dim,
        )
        assert logits.shape == (batch_size, self.forecast_horizon, self.num_features, num_experts)
        assert weights.shape == (batch_size, self.forecast_horizon, self.num_features, num_experts)
        assert mixed_prediction.shape == (batch_size, self.forecast_horizon, self.num_features)
        assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights[..., 0]), atol=1e-6)

        if return_intermediates:
            return mixed_prediction, weights, logits, {
                "projected_history": projected_history,
                "encoded_history": encoded_history,
                "horizon_queries": horizon_queries,
                "horizon_history": horizon_history,
                "horizon_variable_features": horizon_variable_features,
                "prediction_features": prediction_features,
                "prediction_embedding": prediction_embedding,
                "fused_features": fused_features,
                "router_features": router_features,
                "expert_vectors": expert_vectors,
                "logits": logits,
                "weights": weights,
                "mixed_prediction": mixed_prediction,
            }
        return mixed_prediction, weights, logits


ROUTER2_SOFT_ROUTER_CLASSES = {
    "original": Router2FeatureRouter,
    "multiscale_tcn_expert_embeddings": MultiscaleTCNExpertEmbeddingRouter,
}


def make_router2(router_type: str, num_experts: int) -> nn.Module:
    try:
        router_class = ROUTER2_SOFT_ROUTER_CLASSES[router_type]
    except KeyError as exc:
        raise ValueError(f"ROUTER_TYPE must be one of {tuple(ROUTER2_SOFT_ROUTER_CLASSES)}") from exc
    return router_class(num_experts=num_experts)


def router2_from_config(config: dict, fallback_router_type: str, fallback_num_experts: int) -> nn.Module:
    config = dict(config)
    router_type = config.pop("router_type", fallback_router_type)
    config.setdefault("num_experts", fallback_num_experts)
    try:
        router_class = ROUTER2_SOFT_ROUTER_CLASSES[router_type]
    except KeyError as exc:
        raise ValueError(f"Saved router type {router_type!r} is not available") from exc
    return router_class(**config)


def router2_checkpoint_path(checkpoint_dir: Union[str, Path], router_type: str) -> Path:
    checkpoint_dir = _resolve_project_path(checkpoint_dir)
    if router_type == "original":
        return checkpoint_dir / "best_router2.pt"
    return checkpoint_dir / f"best_router2_{router_type}.pt"


def router2_results_paths(output_dir: Union[str, Path], router_type: str) -> Tuple[Path, Path]:
    output_dir = _resolve_project_path(output_dir)
    if router_type == "original":
        return output_dir / "router2_test_comparison.csv", output_dir / "router2_test_metrics.json"
    return (
        output_dir / f"router2_{router_type}_test_comparison.csv",
        output_dir / f"router2_{router_type}_test_metrics.json",
    )


def run_router2_dummy_forward_test(router_type: str = "multiscale_tcn_expert_embeddings", num_experts: int = 3, batch_size: int = 4, device: str = "cpu"):
    device = torch.device(device)
    router = make_router2(router_type, num_experts=num_experts).to(device)
    router.train()
    history = torch.randn(batch_size, DEFAULT_INPUT_LEN, DEFAULT_NUM_FEATURES, device=device)
    if getattr(router, "expert_prediction_layout", "expert_feature") == "feature_expert":
        expert_predictions = torch.randn(batch_size, DEFAULT_OUTPUT_LEN, DEFAULT_NUM_FEATURES, num_experts, device=device)
    else:
        expert_predictions = torch.randn(batch_size, DEFAULT_OUTPUT_LEN, num_experts, DEFAULT_NUM_FEATURES, device=device)
    if hasattr(router, "expert_embeddings"):
        mixed_prediction, weights, logits, intermediates = router(
            history,
            expert_predictions,
            return_intermediates=True,
        )
    else:
        mixed_prediction, weights, logits = router(history, expert_predictions)
        intermediates = {}
    loss = mixed_prediction.square().mean()
    loss.backward()
    assert any(parameter.grad is not None for parameter in router.parameters())
    if hasattr(router, "expert_embeddings"):
        assert router.expert_embeddings.grad is not None
    print("Router 2 dummy forward-pass test")
    print("router_type:", router_type)
    print("history:", tuple(history.shape))
    print("expert_predictions:", tuple(expert_predictions.shape))
    for name, tensor in intermediates.items():
        print(f"{name}:", tuple(tensor.shape))
    print("logits:", tuple(logits.shape))
    print("weights:", tuple(weights.shape))
    print("weight sum range:", float(weights.sum(dim=-1).min()), "to", float(weights.sum(dim=-1).max()))
    print("mixed_prediction:", tuple(mixed_prediction.shape))
    if hasattr(router, "expert_embeddings"):
        print("expert_embeddings grad:", router.expert_embeddings.grad is not None)
    return router

## Router 2 Helpers

These helpers keep the same split and frozen-expert rules as the original router notebook, but use `Router2FeatureRouter` for forward, training, checkpointing, and evaluation.

In [3]:
def _router2_expert_prediction(router: nn.Module, expert_predictions: torch.Tensor, expert_index: int) -> torch.Tensor:
    if getattr(router, "expert_prediction_layout", "expert_feature") == "feature_expert":
        return expert_predictions[..., expert_index]
    return expert_predictions[:, :, expert_index, :]


def _router2_equal_prediction(router: nn.Module, expert_predictions: torch.Tensor) -> torch.Tensor:
    if getattr(router, "expert_prediction_layout", "expert_feature") == "feature_expert":
        return expert_predictions.mean(dim=-1)
    return expert_predictions.mean(dim=2)


def router2_forward(
    router: nn.Module,
    experts: Sequence[nn.Module],
    inputs: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if len(experts) != router.num_experts:
        raise ValueError(f"Router 2 expects {router.num_experts} experts, got {len(experts)}")
    layout = getattr(router, "expert_prediction_layout", "expert_feature")
    with torch.no_grad():
        expert_predictions = torch.stack(
            [_call_expert_model(expert, inputs).detach() for expert in experts],
            dim=-1 if layout == "feature_expert" else 2,
        )
    mixed_prediction, router_weights, router_scores = router(inputs, expert_predictions)
    expected_forecast = (inputs.shape[0], router.forecast_horizon, router.num_features)
    if layout == "feature_expert":
        expected_stack = (inputs.shape[0], router.forecast_horizon, router.num_features, router.num_experts)
        expected_weights = expected_stack
    else:
        expected_stack = (inputs.shape[0], router.forecast_horizon, router.num_experts, router.num_features)
        expected_weights = (inputs.shape[0], router.forecast_horizon, router.num_experts)
    assert tuple(expert_predictions.shape) == expected_stack
    assert tuple(mixed_prediction.shape) == expected_forecast
    assert tuple(router_weights.shape) == expected_weights
    assert tuple(router_scores.shape) == expected_weights
    assert torch.allclose(router_weights.sum(dim=-1), torch.ones_like(router_weights[..., 0]), atol=1e-6)
    return expert_predictions, router_weights, mixed_prediction


def evaluate_router2_pipeline(
    router: nn.Module,
    experts: Sequence[nn.Module],
    loader,
    device: Union[str, torch.device] = "cpu",
    scaler=None,
    print_shapes: bool = False,
    expert_names: Optional[Sequence[str]] = None,
) -> dict:
    device = torch.device(device)
    expert_names = tuple(expert_names or [f"Expert {index + 1}" for index in range(len(experts))])
    if len(expert_names) != len(experts):
        raise ValueError("expert_names must match the selected experts")
    if router.num_experts != len(experts):
        raise ValueError(f"Router 2 expects {router.num_experts} experts, got {len(experts)}")
    router.to(device)
    router.eval()
    assert_experts_frozen(*experts)
    for expert in experts:
        expert.to(device)
        expert.eval()

    totals = {"router_abs": 0.0, "router_sq": 0.0, "count": 0}
    totals.update({expert_name: 0.0 for expert_name in expert_names})
    expert_weight_sums = {expert_name: 0.0 for expert_name in expert_names}
    weight_count = 0
    with torch.no_grad():
        for batch_index, batch in enumerate(loader):
            inputs, targets, targets_mask = _prepare_forecasting_batch(batch, device, scaler)
            expert_predictions, router_weights, mixed_prediction = router2_forward(router, experts, inputs)
            _check_shapes(mixed_prediction, targets, getattr(router, "display_name", "Router 2 feature router"))
            if print_shapes and batch_index == 0:
                print("\nFirst Router 2 validation batch")
                print("Input shape:", list(inputs.shape))
                print("Target shape:", list(targets.shape))
                print("Expert prediction stack shape:", list(expert_predictions.shape))
                for expert_index, expert_name in enumerate(expert_names):
                    print(f"{expert_name} prediction shape:", list(_router2_expert_prediction(router, expert_predictions, expert_index).shape))
                print("Router 2 weight shape:", list(router_weights.shape))
                print("Router 2 mixed shape:", list(mixed_prediction.shape))
            abs_sum, sq_sum, count = _accumulate_errors(mixed_prediction, targets, targets_mask)
            for expert_index, expert_name in enumerate(expert_names):
                expert_abs, _, _ = _accumulate_errors(_router2_expert_prediction(router, expert_predictions, expert_index), targets, targets_mask)
                totals[expert_name] += expert_abs
            totals["router_abs"] += abs_sum
            totals["router_sq"] += sq_sum
            totals["count"] += count
            for expert_index, expert_name in enumerate(expert_names):
                expert_weight_sums[expert_name] += router_weights[..., expert_index].sum().item()
            weight_count += router_weights[..., 0].numel()
    if totals["count"] == 0 or weight_count == 0:
        raise ValueError("Router 2 evaluation produced no elements")
    _assert_no_expert_gradients(*experts)
    return {
        "mae": totals["router_abs"] / totals["count"],
        "mse": totals["router_sq"] / totals["count"],
        "expert_mae": {expert_name: totals[expert_name] / totals["count"] for expert_name in expert_names},
        "average_expert_weights": {expert_name: expert_weight_sums[expert_name] / weight_count for expert_name in expert_names},
    }

In [4]:
def train_router2_model(
    router: nn.Module,
    experts: Sequence[nn.Module],
    optimizer: torch.optim.Optimizer,
    train_loader,
    val_loader,
    checkpoint_path: Union[str, Path],
    max_epochs: int,
    patience: int = 10,
    device: Union[str, torch.device] = "cpu",
    scaler=None,
    dataset_config: Optional[dict] = None,
    expert_checkpoint_paths: Optional[dict] = None,
    expert_names: Optional[Sequence[str]] = None,
) -> Tuple[dict, ...]:
    if getattr(train_loader.dataset, "split_role", None) != "router_train":
        raise ValueError("Router 2 training requires router_train")
    if getattr(val_loader.dataset, "split_role", None) != "router_val":
        raise ValueError("Router 2 validation requires router_val")
    expert_names = tuple(expert_names or [f"Expert {index + 1}" for index in range(len(experts))])
    if len(expert_names) != len(experts):
        raise ValueError("expert_names must match the selected experts")
    if router.num_experts != len(experts):
        raise ValueError(f"Router 2 expects {router.num_experts} experts, got {len(experts)}")
    device = torch.device(device)
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    assert_experts_frozen(*experts)
    expert_parameter_ids = {id(p) for expert in experts for p in expert.parameters()}
    optimizer_parameter_ids = {id(p) for group in optimizer.param_groups for p in group["params"]}
    router_parameter_ids = {id(p) for p in router.parameters()}
    if optimizer_parameter_ids & expert_parameter_ids:
        raise ValueError("Router 2 optimizer contains expert parameters")
    if not optimizer_parameter_ids or not optimizer_parameter_ids.issubset(router_parameter_ids):
        raise ValueError("Router 2 optimizer must contain only router parameters")

    for expert in experts:
        expert.to(device)
        expert.eval()
        for parameter in expert.parameters():
            parameter.grad = None
    router.to(device)
    best_validation_mae = float("inf")
    epochs_without_improvement = 0
    history = []
    loss_function = nn.SmoothL1Loss(reduction="none")

    for epoch in range(1, max_epochs + 1):
        router.train()
        abs_total = 0.0
        loss_total = 0.0
        element_count = 0
        expert_weight_sums = {expert_name: 0.0 for expert_name in expert_names}
        weight_count = 0
        for batch_index, batch in enumerate(train_loader):
            inputs, targets, targets_mask = _prepare_forecasting_batch(batch, device, scaler)
            optimizer.zero_grad(set_to_none=True)
            expert_predictions, router_weights, mixed_prediction = router2_forward(router, experts, inputs)
            _check_shapes(mixed_prediction, targets, getattr(router, "display_name", "Router 2 feature router"))
            loss_values = loss_function(mixed_prediction, targets)
            loss = loss_values[targets_mask].mean()
            loss.backward()
            if not any(parameter.grad is not None for parameter in router.parameters()):
                raise RuntimeError("No Router 2 parameter received a gradient")
            if hasattr(router, "expert_embeddings") and router.expert_embeddings.grad is None:
                raise RuntimeError("Router expert embeddings did not receive a gradient")
            _assert_no_expert_gradients(*experts)
            optimizer.step()
            abs_sum, _, count = _accumulate_errors(mixed_prediction.detach(), targets, targets_mask)
            abs_total += abs_sum
            loss_total += loss.detach().item() * count
            element_count += count
            for expert_index, expert_name in enumerate(expert_names):
                expert_weight_sums[expert_name] += router_weights[..., expert_index].detach().sum().item()
            weight_count += router_weights[..., 0].numel()
            if epoch == 1 and batch_index == 0:
                print("\nFirst Router 2 training batch")
                print("Router type:", getattr(router, "router_type", "original"))
                print("Input shape:", list(inputs.shape))
                print("Target shape:", list(targets.shape))
                print("Expert prediction stack shape:", list(expert_predictions.shape))
                for expert_index, expert_name in enumerate(expert_names):
                    print(f"{expert_name} prediction shape:", list(_router2_expert_prediction(router, expert_predictions, expert_index).shape))
                print("Router 2 weight shape:", list(router_weights.shape))
                print("Router 2 mixed shape:", list(mixed_prediction.shape))

        validation = evaluate_router2_pipeline(router, experts, val_loader, device=device, scaler=scaler, print_shapes=(epoch == 1), expert_names=expert_names)
        checkpoint_saved = validation["mae"] < best_validation_mae
        if checkpoint_saved:
            best_validation_mae = validation["mae"]
            epochs_without_improvement = 0
            torch.save(
                {
                    "router_state_dict": router.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "training_loss": loss_total / element_count,
                    "validation_mae": validation["mae"],
                    "validation_mse": validation["mse"],
                    "average_expert_weights": {
                        expert_name: expert_weight_sums[expert_name] / weight_count
                        for expert_name in expert_names
                    },
                    "selected_expert_names": list(expert_names),
                    "router_config": router.config_dict(),
                    "dataset_config": dict(dataset_config or {}),
                    "expert_checkpoint_paths": {name: str(path) for name, path in (expert_checkpoint_paths or {}).items()},
                    **({"scaler_stats": scaler.stats} if scaler is not None else {}),
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        row = {
            "epoch": epoch,
            "training_loss": loss_total / element_count,
            "train_mae": abs_total / element_count,
            "validation_mae": validation["mae"],
            "validation_mse": validation["mse"],
            "average_expert_weights": {
                expert_name: expert_weight_sums[expert_name] / weight_count
                for expert_name in expert_names
            },
            "checkpoint_saved": checkpoint_saved,
            "early_stopping_counter": epochs_without_improvement,
        }
        history.append(row)
        print(
            f"Router 2 epoch {epoch:>3d}/{max_epochs}: "
            f"train MAE={row['train_mae']:.6f}, val MAE={row['validation_mae']:.6f}, "
            "avg weights=(" + ", ".join(
                f"{expert_name}={row['average_expert_weights'][expert_name]:.3f}"
                for expert_name in expert_names
            ) + "), "
            f"checkpoint saved={checkpoint_saved}"
        )
        if epochs_without_improvement >= patience:
            print(f"Router 2 early stopping after {epoch} epochs")
            break
    selected = _load_torch_checkpoint(checkpoint_path, device)
    router.load_state_dict(selected["router_state_dict"])
    router.eval()
    return tuple(history)

In [5]:
def _resolve_project_path(path_value):
    path = Path(path_value)
    if path.is_absolute():
        return path
    return ROOT / path


def _load_router2_environment(data_dir, checkpoint_dir, batch_size, device, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    data_dir = _resolve_project_path(data_dir)
    checkpoint_dir = _resolve_project_path(checkpoint_dir)
    full_data = load_full_chronological_data(data_dir)
    _assert_full_data_contract(full_data, DEFAULT_NUM_FEATURES)
    loaders, scaler = prepare_chronological_dataloaders(
        full_data=full_data,
        scaler=ZScoreScaler(norm_each_channel=True, rescale=False),
        batch_size=batch_size,
        input_len=DEFAULT_INPUT_LEN,
        output_len=DEFAULT_OUTPUT_LEN,
    )
    experts, expert_names, _, expert_checkpoint_paths = build_selected_candidate_experts(
        checkpoint_dir=checkpoint_dir,
        device=torch.device(device),
        scaler=scaler,
    )
    print("Selected Router 2 experts:", ", ".join(expert_names))
    return full_data, loaders, scaler, experts, expert_names, expert_checkpoint_paths


def run_router2_input_check_stage(data_dir="datasets/ETTh1", checkpoint_dir="checkpoints", batch_size=512, device="cpu", seed=7, router_type="original"):
    _, loaders, scaler, experts, expert_names, _ = _load_router2_environment(data_dir, checkpoint_dir, batch_size, device, seed)
    router = make_router2(router_type, num_experts=len(experts)).to(device)
    batch = next(iter(loaders["router_train"]))
    inputs, _, _ = _prepare_forecasting_batch(batch, torch.device(device), scaler)
    with torch.no_grad():
        expert_predictions, router_weights, mixed_prediction = router2_forward(router, experts, inputs)
    print("Router 2 input check")
    print("router_type:", router_type)
    print("inputs:", tuple(inputs.shape))
    print("expert_predictions:", tuple(expert_predictions.shape))
    for expert_index, expert_name in enumerate(expert_names):
        print(f"{expert_name} prediction:", tuple(_router2_expert_prediction(router, expert_predictions, expert_index).shape))
    print("router_weights:", tuple(router_weights.shape))
    print("mixed_prediction:", tuple(mixed_prediction.shape))
    return router


def run_router2_training_stage(data_dir="datasets/ETTh1", checkpoint_dir="checkpoints", batch_size=512, max_epochs=50, patience=10, learning_rate=1e-3, device="cpu", seed=7, router_type="original"):
    full_data, loaders, scaler, experts, expert_names, expert_checkpoint_paths = _load_router2_environment(data_dir, checkpoint_dir, batch_size, device, seed)
    checkpoint_dir = _resolve_project_path(checkpoint_dir)
    run_router2_dummy_forward_test(router_type=router_type, num_experts=len(experts), batch_size=4, device=device)
    router = make_router2(router_type, num_experts=len(experts)).to(device)
    optimizer = torch.optim.Adam(router.parameters(), lr=learning_rate)
    return train_router2_model(
        router=router,
        experts=experts,
        optimizer=optimizer,
        train_loader=loaders["router_train"],
        val_loader=loaders["router_val"],
        checkpoint_path=router2_checkpoint_path(checkpoint_dir, router_type),
        max_epochs=max_epochs,
        patience=patience,
        device=device,
        scaler=scaler,
        dataset_config=_dataset_config_summary(len(full_data)),
        expert_checkpoint_paths=expert_checkpoint_paths,
        expert_names=expert_names,
    )

In [6]:
def evaluate_router2_and_baselines(router, experts, router_val_loader, test_loader, output_dir, device="cpu", scaler=None, expert_names=None):
    if getattr(router_val_loader.dataset, "split_role", None) != "router_val":
        raise ValueError("Router 2 baseline selection requires router_val")
    if getattr(test_loader.dataset, "split_role", None) != "test":
        raise ValueError("Router 2 final evaluation requires test")
    device = torch.device(device)
    expert_names = tuple(expert_names or [f"Expert {index + 1}" for index in range(len(experts))])
    if len(expert_names) != len(experts):
        raise ValueError("expert_names must match the selected experts")
    if router.num_experts != len(experts):
        raise ValueError(f"Router 2 expects {router.num_experts} experts, got {len(experts)}")
    validation = evaluate_router2_pipeline(router, experts, router_val_loader, device=device, scaler=scaler, expert_names=expert_names)
    epsilon = 1e-6
    inverse_scores = {
        expert_name: 1.0 / (validation["expert_mae"][expert_name] + epsilon)
        for expert_name in expert_names
    }
    inverse_total = sum(inverse_scores.values())
    fixed_soft_weights = {
        expert_name: inverse_scores[expert_name] / inverse_total
        for expert_name in expert_names
    }
    globally_best_expert = min(expert_names, key=lambda expert_name: validation["expert_mae"][expert_name])
    router_method_name = getattr(router, "display_name", "Router 2 feature router")
    method_names = (
        *expert_names,
        "Fixed equal average",
        "Fixed validation-based soft weights",
        "Validation-selected best expert",
        router_method_name,
    )
    totals = {name: {"absolute": 0.0, "squared": 0.0, "count": 0} for name in method_names}
    router.eval()
    for expert in experts:
        expert.eval()
    with torch.no_grad():
        for batch in test_loader:
            inputs, targets, targets_mask = _prepare_forecasting_batch(batch, device, scaler)
            expert_predictions, _, router_prediction = router2_forward(router, experts, inputs)
            predictions = {
                expert_name: _router2_expert_prediction(router, expert_predictions, expert_index)
                for expert_index, expert_name in enumerate(expert_names)
            }
            equal_prediction = _router2_equal_prediction(router, expert_predictions)
            fixed_soft_prediction = torch.zeros_like(equal_prediction)
            for expert_index, expert_name in enumerate(expert_names):
                fixed_soft_prediction += fixed_soft_weights[expert_name] * _router2_expert_prediction(router, expert_predictions, expert_index)
            best_expert_index = expert_names.index(globally_best_expert)
            predictions.update(
                {
                    "Fixed equal average": equal_prediction,
                    "Fixed validation-based soft weights": fixed_soft_prediction,
                    "Validation-selected best expert": _router2_expert_prediction(router, expert_predictions, best_expert_index),
                    router_method_name: router_prediction,
                }
            )
            for name, prediction in predictions.items():
                _check_shapes(prediction, targets, name)
                abs_sum, sq_sum, count = _accumulate_errors(prediction, targets, targets_mask)
                totals[name]["absolute"] += abs_sum
                totals[name]["squared"] += sq_sum
                totals[name]["count"] += count
    comparison = []
    for name in method_names:
        mae = totals[name]["absolute"] / totals[name]["count"]
        mse = totals[name]["squared"] / totals[name]["count"]
        comparison.append({"Method": name, "Test MAE": mae, "Test MSE": mse, "Test RMSE": mse ** 0.5})
    comparison.sort(key=lambda row: row["Test MAE"])
    router_row = next(row for row in comparison if row["Method"] == router_method_name)
    strongest_baseline = min((row for row in comparison if row["Method"] != router_method_name), key=lambda row: row["Test MAE"])
    results = {
        "router_type": getattr(router, "router_type", "original"),
        "selected_expert_names": list(expert_names),
        "comparison": comparison,
        "router2": router_row,
        "strongest_baseline": strongest_baseline,
        "absolute_mae_improvement": strongest_baseline["Test MAE"] - router_row["Test MAE"],
        "router_validation_baseline_selection": {
            "expert_validation_mae": validation["expert_mae"],
            "fixed_soft_weights": fixed_soft_weights,
            "globally_best_expert": globally_best_expert,
        },
    }
    output_dir = _resolve_project_path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path, json_path = router2_results_paths(output_dir, getattr(router, "router_type", "original"))
    with csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=("Method", "Test MAE", "Test MSE", "Test RMSE"))
        writer.writeheader()
        writer.writerows(comparison)
    with json_path.open("w", encoding="utf-8") as file:
        json.dump(results, file, indent=2)
    print("\nRouter 2 final comparison")
    for row in comparison:
        print(f"{row['Method']:<45} MAE={row['Test MAE']:.6f} MSE={row['Test MSE']:.6f}")
    print("Saved:", csv_path)
    print("Saved:", json_path)
    return results


def run_router2_test_stage(data_dir="datasets/ETTh1", checkpoint_dir="checkpoints", output_dir="results", batch_size=512, device="cpu", seed=7, router_type="original"):
    _, loaders, scaler, experts, expert_names, _ = _load_router2_environment(data_dir, checkpoint_dir, batch_size, device, seed)
    checkpoint = _load_torch_checkpoint(router2_checkpoint_path(checkpoint_dir, router_type), torch.device(device))
    router_config = dict(checkpoint["router_config"])
    router = router2_from_config(router_config, fallback_router_type=router_type, fallback_num_experts=len(experts)).to(device)
    router.load_state_dict(checkpoint["router_state_dict"])
    router.eval()
    return evaluate_router2_and_baselines(router, experts, loaders["router_val"], loaders["test"], output_dir=output_dir, device=device, scaler=scaler, expert_names=expert_names)

## RouterDC Hard Router

This is a separate hard-routing path. It does not consume expert predictions, disagreement features, or soft per-horizon/per-variable weights. It embeds each input window, compares that window embedding with trainable expert embeddings, selects one expert for the whole window, then runs only the selected expert for each sample.

In [7]:
class RouterDCHardRouter(nn.Module):
    """History-only RouterDC-style hard router that selects one expert per window."""

    router_type = "routerdc_hard"

    def __init__(
        self,
        input_len: int = DEFAULT_INPUT_LEN,
        num_features: int = DEFAULT_NUM_FEATURES,
        num_experts: int = 2,
        embedding_dim: int = 64,
        hidden_dim: int = 64,
        dropout: float = 0.1,
        router_temperature: float = 1.0,
    ) -> None:
        super().__init__()
        self.input_len = input_len
        self.num_features = num_features
        if num_experts < 2:
            raise ValueError("RouterDCHardRouter requires at least two selected experts")
        self.num_experts = num_experts
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.router_temperature = router_temperature

        self.input_projection = nn.Sequential(
            nn.Linear(num_features, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
        )
        self.temporal_encoder = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GELU(),
        )
        self.window_projection = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, embedding_dim),
        )
        self.expert_embeddings = nn.Parameter(
            torch.randn(num_experts, embedding_dim)
        )
        nn.init.normal_(self.expert_embeddings, mean=0.0, std=0.02)

    def config_dict(self) -> dict:
        return {
            "router_type": self.router_type,
            "input_len": self.input_len,
            "num_features": self.num_features,
            "num_experts": self.num_experts,
            "embedding_dim": self.embedding_dim,
            "hidden_dim": self.hidden_dim,
            "dropout": self.dropout,
            "router_temperature": self.router_temperature,
        }

    def encode(self, history: torch.Tensor) -> torch.Tensor:
        batch_size = history.shape[0]
        assert history.shape[1:] == (self.input_len, self.num_features)
        projected = self.input_projection(history)
        encoded = self.temporal_encoder(projected.transpose(1, 2)).transpose(1, 2)
        window_embedding = self.window_projection(encoded.mean(dim=1))
        assert window_embedding.shape == (batch_size, self.embedding_dim)
        return window_embedding

    def router_log_probabilities(self, history: torch.Tensor):
        batch_size = history.shape[0]
        window_embedding = self.encode(history)
        q = F.normalize(window_embedding, p=2, dim=-1)
        k = F.normalize(self.expert_embeddings, p=2, dim=-1)
        similarities = q @ k.T
        router_log_probs = F.log_softmax(
            similarities / self.router_temperature,
            dim=-1,
        )
        router_probs = router_log_probs.exp()
        assert similarities.shape == (batch_size, self.num_experts)
        assert router_log_probs.shape == (batch_size, self.num_experts)
        assert router_probs.shape == (batch_size, self.num_experts)
        assert torch.allclose(router_probs.sum(dim=-1), torch.ones(batch_size, device=history.device), atol=1e-6)
        return window_embedding, similarities, router_log_probs, router_probs

    def forward(self, history: torch.Tensor):
        _, similarities, router_log_probs, router_probs = self.router_log_probabilities(history)
        selected_expert = similarities.argmax(dim=-1)
        assert selected_expert.shape == (history.shape[0],)
        return similarities, router_log_probs, router_probs, selected_expert


def routerdc_hard_checkpoint_path(checkpoint_dir: Union[str, Path], use_window_contrastive_loss: bool) -> Path:
    suffix = "contrastive" if use_window_contrastive_loss else "no_contrastive"
    return Path(checkpoint_dir) / f"best_routerdc_hard_{suffix}.pt"


def routerdc_hard_cache_path(checkpoint_dir: Union[str, Path], split_role: str) -> Path:
    return Path(checkpoint_dir) / f"routerdc_hard_{split_role}_expert_error_cache.pt"


def run_routerdc_hard_dummy_forward_test(num_experts: int = 3, batch_size: int = 4, device: str = "cpu"):
    device = torch.device(device)
    router = RouterDCHardRouter(num_experts=num_experts).to(device)
    router.train()
    history = torch.randn(batch_size, DEFAULT_INPUT_LEN, DEFAULT_NUM_FEATURES, device=device)
    similarities, router_log_probs, router_probs, selected_expert = router(history)
    loss = -router_log_probs.mean()
    loss.backward()
    assert similarities.shape == (batch_size, num_experts)
    assert router_log_probs.shape == (batch_size, num_experts)
    assert router_probs.shape == (batch_size, num_experts)
    assert selected_expert.shape == (batch_size,)
    assert router.expert_embeddings.grad is not None
    assert router.input_projection[0].weight.grad is not None
    assert any(parameter.grad is not None for parameter in router.temporal_encoder.parameters())
    print("RouterDC hard dummy forward-pass test")
    print("history:", tuple(history.shape))
    print("window_embedding:", tuple(router.encode(history.detach()).shape))
    print("similarities:", tuple(similarities.shape))
    print("router_log_probs:", tuple(router_log_probs.shape))
    print("router_probs:", tuple(router_probs.shape))
    print("selected_expert:", tuple(selected_expert.shape))
    print("probability sum range:", float(router_probs.sum(dim=-1).min()), "to", float(router_probs.sum(dim=-1).max()))
    print("expert_embeddings grad:", router.expert_embeddings.grad is not None)
    return router


def build_routerdc_expert_performance_cache(
    experts: Sequence[nn.Module],
    loader,
    device: Union[str, torch.device] = "cpu",
    scaler=None,
    expert_names: Optional[Sequence[str]] = None,
    error_temperature: float = 0.1,
    cache_path: Optional[Union[str, Path]] = None,
    force_rebuild: bool = False,
) -> dict:
    device = torch.device(device)
    split_role = getattr(loader.dataset, "split_role", None)
    if split_role not in {"router_train", "router_val", "test"}:
        raise ValueError("RouterDC cache must be built from router_train, router_val, or test")
    expert_names = tuple(expert_names or [f"Expert {index + 1}" for index in range(len(experts))])
    cache_path = Path(cache_path) if cache_path is not None else None
    if cache_path is not None and cache_path.exists() and not force_rebuild:
        cached = _load_torch_checkpoint(cache_path, torch.device("cpu"))
        if (
            cached.get("split_role") == split_role
            and tuple(cached.get("expert_names", ())) == expert_names
            and cached.get("num_windows") == len(loader.dataset)
            and abs(float(cached.get("error_temperature", error_temperature)) - float(error_temperature)) < 1e-12
        ):
            return cached

    assert_experts_frozen(*experts)
    for expert in experts:
        expert.to(device)
        expert.eval()

    error_rows = []
    history_rows = []
    sample_index_rows = []
    absolute_index_rows = []
    cursor = 0
    with torch.no_grad():
        for batch in loader:
            inputs, targets, _ = _prepare_forecasting_batch(batch, device, scaler)
            batch_errors = []
            for expert in experts:
                prediction = _call_expert_model(expert, inputs).detach()
                _check_shapes(prediction, targets, "RouterDC cache expert prediction")
                batch_errors.append(torch.mean(torch.abs(prediction - targets), dim=(1, 2)))
            batch_error_matrix = torch.stack(batch_errors, dim=1)
            batch_size = inputs.shape[0]
            sample_indices = torch.arange(cursor, cursor + batch_size)
            absolute_indices = sample_indices + int(getattr(loader.dataset.boundary, "start", 0))
            error_rows.append(batch_error_matrix.cpu())
            history_rows.append(inputs.detach().cpu())
            sample_index_rows.append(sample_indices)
            absolute_index_rows.append(absolute_indices)
            cursor += batch_size

    error_matrix = torch.cat(error_rows, dim=0)
    histories = torch.cat(history_rows, dim=0)
    sample_indices = torch.cat(sample_index_rows, dim=0)
    absolute_start_indices = torch.cat(absolute_index_rows, dim=0)
    target_probs = torch.softmax(-error_matrix / error_temperature, dim=-1)
    assert error_matrix.shape == (len(loader.dataset), len(experts))
    assert target_probs.shape == (len(loader.dataset), len(experts))
    assert torch.allclose(target_probs.sum(dim=-1), torch.ones(len(loader.dataset)), atol=1e-6)
    cache = {
        "split_role": split_role,
        "expert_names": expert_names,
        "num_windows": len(loader.dataset),
        "error_temperature": float(error_temperature),
        "error_matrix": error_matrix,
        "target_probs": target_probs,
        "best_expert": error_matrix.argmin(dim=-1),
        "histories": histories,
        "sample_indices": sample_indices,
        "absolute_start_indices": absolute_start_indices,
    }
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(cache, cache_path)
    _assert_no_expert_gradients(*experts)
    return cache


def build_routerdc_window_cluster_cache(
    train_cache: dict,
    num_clusters: int = 16,
    pca_components: int = 16,
    kmeans_iterations: int = 50,
    seed: int = 7,
) -> dict:
    if train_cache["split_role"] != "router_train":
        raise ValueError("Window contrastive clusters must be fit on router_train only")
    histories = train_cache["histories"].reshape(train_cache["histories"].shape[0], -1).to(torch.float32)
    feature_mean = histories.mean(dim=0, keepdim=True)
    feature_std = histories.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
    standardized = (histories - feature_mean) / feature_std
    max_components = max(1, min(pca_components, standardized.shape[0] - 1, standardized.shape[1]))
    _, _, vh = torch.linalg.svd(standardized, full_matrices=False)
    components = vh[:max_components].T.contiguous()
    pca_features = standardized @ components
    cluster_count = max(1, min(num_clusters, pca_features.shape[0]))
    generator = torch.Generator().manual_seed(seed)
    initial_indices = torch.randperm(pca_features.shape[0], generator=generator)[:cluster_count]
    centroids = pca_features[initial_indices].clone()
    labels = torch.zeros(pca_features.shape[0], dtype=torch.long)
    for _ in range(kmeans_iterations):
        distances = torch.cdist(pca_features, centroids)
        labels = distances.argmin(dim=1)
        updated = []
        for cluster_index in range(cluster_count):
            members = pca_features[labels == cluster_index]
            updated.append(members.mean(dim=0) if members.numel() else centroids[cluster_index])
        next_centroids = torch.stack(updated, dim=0)
        if torch.allclose(next_centroids, centroids):
            centroids = next_centroids
            break
        centroids = next_centroids
    assert labels.shape == (train_cache["num_windows"],)
    return {
        "cluster_labels": labels,
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "pca_components": components,
        "centroids": centroids,
        "num_clusters": cluster_count,
    }


def supervised_window_contrastive_loss(
    window_embeddings: torch.Tensor,
    cluster_labels: torch.Tensor,
    contrastive_temperature: float = 0.2,
) -> torch.Tensor:
    batch_size = window_embeddings.shape[0]
    if batch_size <= 1:
        return window_embeddings.sum() * 0.0
    embeddings = F.normalize(window_embeddings, p=2, dim=-1)
    logits = embeddings @ embeddings.T / contrastive_temperature
    eye = torch.eye(batch_size, dtype=torch.bool, device=window_embeddings.device)
    positive_mask = cluster_labels.unsqueeze(0).eq(cluster_labels.unsqueeze(1)) & ~eye
    valid_rows = positive_mask.sum(dim=1) > 0
    if not valid_rows.any():
        return window_embeddings.sum() * 0.0
    logits = logits.masked_fill(eye, -torch.inf)
    log_probs = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    positive_log_probs = torch.where(
        positive_mask,
        log_probs,
        torch.zeros_like(log_probs),
    )
    row_loss = -positive_log_probs.sum(dim=1) / positive_mask.sum(dim=1).clamp_min(1)
    return row_loss[valid_rows].mean()


def routerdc_hard_route_and_predict(
    router: RouterDCHardRouter,
    experts: Sequence[nn.Module],
    inputs: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, dict]:
    if len(experts) != router.num_experts:
        raise ValueError(f"RouterDCHardRouter expects {router.num_experts} experts, got {len(experts)}")
    router.eval()
    for expert in experts:
        expert.eval()
    with torch.no_grad():
        similarities, _, router_probs, selected_expert = router(inputs)
        prediction = torch.empty(
            inputs.shape[0],
            DEFAULT_OUTPUT_LEN,
            DEFAULT_NUM_FEATURES,
            device=inputs.device,
            dtype=inputs.dtype,
        )
        expert_call_counts = {}
        for expert_index, expert in enumerate(experts):
            selected_mask = selected_expert == expert_index
            expert_call_counts[expert_index] = int(selected_mask.sum().item())
            if selected_mask.any():
                prediction[selected_mask] = _call_expert_model(expert, inputs[selected_mask]).detach()

    assert similarities.shape == (inputs.shape[0], router.num_experts)
    assert router_probs.shape == (inputs.shape[0], router.num_experts)
    assert selected_expert.shape == (inputs.shape[0],)
    assert prediction.shape == (inputs.shape[0], DEFAULT_OUTPUT_LEN, DEFAULT_NUM_FEATURES)
    _assert_no_expert_gradients(*experts)
    return similarities, router_probs, selected_expert, prediction, expert_call_counts


def _routerdc_cache_slice(cache: dict, start: int, stop: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    target_probs = cache["target_probs"][start:stop].to(device)
    best_expert = cache["best_expert"][start:stop].to(device)
    sample_indices = cache["sample_indices"][start:stop]
    assert target_probs.shape[0] == stop - start
    assert best_expert.shape[0] == stop - start
    assert torch.equal(sample_indices, torch.arange(start, stop))
    return target_probs, best_expert, sample_indices


def evaluate_routerdc_hard_router(
    router: RouterDCHardRouter,
    experts: Sequence[nn.Module],
    loader,
    device: Union[str, torch.device] = "cpu",
    scaler=None,
    expert_names: Optional[Sequence[str]] = None,
    performance_cache: Optional[dict] = None,
    print_shapes: bool = False,
) -> dict:
    device = torch.device(device)
    expert_names = tuple(expert_names or [f"Expert {index + 1}" for index in range(len(experts))])
    router.to(device)
    router.eval()
    assert_experts_frozen(*experts)
    for expert in experts:
        expert.to(device)
        expert.eval()

    totals = {"absolute": 0.0, "squared": 0.0, "count": 0}
    selection_counts = torch.zeros(len(experts), dtype=torch.long)
    router_probability_sum = torch.zeros(len(experts), dtype=torch.float64)
    selected_rows = []
    entropy_sum = 0.0
    window_count = 0
    cursor = 0
    with torch.no_grad():
        for batch_index, batch in enumerate(loader):
            inputs, targets, targets_mask = _prepare_forecasting_batch(batch, device, scaler)
            similarities, router_probs, selected_expert, prediction, call_counts = routerdc_hard_route_and_predict(router, experts, inputs)
            _check_shapes(prediction, targets, "RouterDCHardRouter")
            abs_sum, sq_sum, count = _accumulate_errors(prediction, targets, targets_mask)
            totals["absolute"] += abs_sum
            totals["squared"] += sq_sum
            totals["count"] += count
            selection_counts += torch.bincount(selected_expert.cpu(), minlength=len(experts))
            router_probability_sum += router_probs.detach().cpu().to(torch.float64).sum(dim=0)
            entropy_sum += (-(router_probs * torch.log(router_probs.clamp_min(1e-12))).sum(dim=-1)).sum().item()
            selected_rows.append(selected_expert.cpu())
            window_count += inputs.shape[0]
            if performance_cache is not None:
                _routerdc_cache_slice(performance_cache, cursor, cursor + inputs.shape[0], device)
            cursor += inputs.shape[0]
            if print_shapes and batch_index == 0:
                print("\nFirst RouterDC hard-router inference batch")
                print("history:", tuple(inputs.shape))
                print("similarities:", tuple(similarities.shape))
                print("router_probs:", tuple(router_probs.shape))
                print("selected_expert:", tuple(selected_expert.shape))
                print("prediction:", tuple(prediction.shape))
                print("selected expert call counts:", {expert_names[index]: count for index, count in call_counts.items()})

    selected_expert_indices = torch.cat(selected_rows, dim=0)
    result = {
        "mae": totals["absolute"] / totals["count"],
        "mse": totals["squared"] / totals["count"],
        "selection_percentage": {
            expert_name: 100.0 * selection_counts[index].item() / max(window_count, 1)
            for index, expert_name in enumerate(expert_names)
        },
        "average_router_probabilities": {
            expert_name: (router_probability_sum[index] / max(window_count, 1)).item()
            for index, expert_name in enumerate(expert_names)
        },
        "routing_entropy": entropy_sum / max(window_count, 1),
        "selected_expert_indices": selected_expert_indices,
    }
    if performance_cache is not None:
        error_matrix = performance_cache["error_matrix"]
        best_expert = performance_cache["best_expert"]
        target_probs = performance_cache["target_probs"]
        result.update(
            {
                "individual_expert_mae": {
                    expert_name: error_matrix[:, index].mean().item()
                    for index, expert_name in enumerate(expert_names)
                },
                "oracle_mae": error_matrix.min(dim=1).values.mean().item(),
                "lowest_error_selection_accuracy": (selected_expert_indices == best_expert).to(torch.float32).mean().item(),
                "average_target_suitability_probabilities": {
                    expert_name: target_probs[:, index].mean().item()
                    for index, expert_name in enumerate(expert_names)
                },
            }
        )
    expert_vectors = F.normalize(router.expert_embeddings.detach().cpu(), p=2, dim=-1)
    result["expert_embedding_cosine_similarity_matrix"] = (expert_vectors @ expert_vectors.T).tolist()
    _assert_no_expert_gradients(*experts)
    return result


def train_routerdc_hard_router(
    router: RouterDCHardRouter,
    experts: Sequence[nn.Module],
    optimizer: torch.optim.Optimizer,
    train_loader,
    val_loader,
    train_cache: dict,
    val_cache: dict,
    checkpoint_path: Union[str, Path],
    max_epochs: int,
    patience: int = 10,
    device: Union[str, torch.device] = "cpu",
    scaler=None,
    expert_names: Optional[Sequence[str]] = None,
    use_window_contrastive_loss: bool = True,
    contrastive_lambda: float = 0.1,
    contrastive_temperature: float = 0.2,
    cluster_cache: Optional[dict] = None,
) -> Tuple[dict, ...]:
    if getattr(train_loader.dataset, "split_role", None) != "router_train":
        raise ValueError("RouterDC hard-router training requires router_train")
    if getattr(val_loader.dataset, "split_role", None) != "router_val":
        raise ValueError("RouterDC hard-router validation requires router_val")
    if train_cache["split_role"] != "router_train" or val_cache["split_role"] != "router_val":
        raise ValueError("RouterDC training requires router_train and router_val caches")
    device = torch.device(device)
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    expert_names = tuple(expert_names or [f"Expert {index + 1}" for index in range(len(experts))])
    assert_experts_frozen(*experts)
    expert_parameter_ids = {id(p) for expert in experts for p in expert.parameters()}
    optimizer_parameter_ids = {id(p) for group in optimizer.param_groups for p in group["params"]}
    router_parameter_ids = {id(p) for p in router.parameters()}
    if optimizer_parameter_ids & expert_parameter_ids:
        raise ValueError("RouterDC optimizer contains expert parameters")
    if not optimizer_parameter_ids or not optimizer_parameter_ids.issubset(router_parameter_ids):
        raise ValueError("RouterDC optimizer must contain only router parameters")
    if use_window_contrastive_loss and cluster_cache is None:
        raise ValueError("Window contrastive loss needs train-only cluster labels")

    for expert in experts:
        expert.to(device)
        expert.eval()
        for parameter in expert.parameters():
            parameter.grad = None
    router.to(device)
    best_validation_mae = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        router.train()
        loss_total = 0.0
        expert_loss_total = 0.0
        contrastive_loss_total = 0.0
        sample_count = 0
        cursor = 0
        for batch_index, batch in enumerate(train_loader):
            inputs, _, _ = _prepare_forecasting_batch(batch, device, scaler)
            batch_size = inputs.shape[0]
            target_probs, _, _ = _routerdc_cache_slice(train_cache, cursor, cursor + batch_size, device)
            window_embedding, similarities, router_log_probs, router_probs = router.router_log_probabilities(inputs)
            window_expert_loss = F.kl_div(
                router_log_probs,
                target_probs,
                reduction="batchmean",
            )
            if use_window_contrastive_loss:
                cluster_labels = cluster_cache["cluster_labels"][cursor:cursor + batch_size].to(device)
                window_window_loss = supervised_window_contrastive_loss(
                    window_embedding,
                    cluster_labels,
                    contrastive_temperature=contrastive_temperature,
                )
            else:
                window_window_loss = window_embedding.sum() * 0.0
            total_loss = window_expert_loss + contrastive_lambda * window_window_loss
            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            if router.expert_embeddings.grad is None:
                raise RuntimeError("RouterDC expert embeddings did not receive gradients")
            if not any(parameter.grad is not None for parameter in router.input_projection.parameters()):
                raise RuntimeError("RouterDC input encoder did not receive gradients")
            if not any(parameter.grad is not None for parameter in router.temporal_encoder.parameters()):
                raise RuntimeError("RouterDC temporal encoder did not receive gradients")
            _assert_no_expert_gradients(*experts)
            optimizer.step()

            loss_total += total_loss.detach().item() * batch_size
            expert_loss_total += window_expert_loss.detach().item() * batch_size
            contrastive_loss_total += window_window_loss.detach().item() * batch_size
            sample_count += batch_size
            cursor += batch_size
            if epoch == 1 and batch_index == 0:
                print("\nFirst RouterDC hard-router training batch")
                print("history:", tuple(inputs.shape))
                print("window_embedding:", tuple(window_embedding.shape))
                print("similarities:", tuple(similarities.shape))
                print("router_log_probs:", tuple(router_log_probs.shape))
                print("router_probs:", tuple(router_probs.shape))
                print("target_probs:", tuple(target_probs.shape))
                print("window_expert_loss:", float(window_expert_loss.detach()))
                print("window_window_loss:", float(window_window_loss.detach()))
        validation = evaluate_routerdc_hard_router(
            router,
            experts,
            val_loader,
            device=device,
            scaler=scaler,
            expert_names=expert_names,
            performance_cache=val_cache,
            print_shapes=(epoch == 1),
        )
        checkpoint_saved = validation["mae"] < best_validation_mae
        if checkpoint_saved:
            best_validation_mae = validation["mae"]
            epochs_without_improvement = 0
            torch.save(
                {
                    "router_state_dict": router.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "training_loss": loss_total / sample_count,
                    "window_expert_loss": expert_loss_total / sample_count,
                    "window_window_loss": contrastive_loss_total / sample_count,
                    "validation_mae": validation["mae"],
                    "validation_mse": validation["mse"],
                    "router_config": router.config_dict(),
                    "selected_expert_names": list(expert_names),
                    "use_window_contrastive_loss": use_window_contrastive_loss,
                    "contrastive_lambda": contrastive_lambda,
                    "contrastive_temperature": contrastive_temperature,
                    "validation_diagnostics": {k: v for k, v in validation.items() if k != "selected_expert_indices"},
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        row = {
            "epoch": epoch,
            "training_loss": loss_total / sample_count,
            "window_expert_loss": expert_loss_total / sample_count,
            "window_window_loss": contrastive_loss_total / sample_count,
            "validation_mae": validation["mae"],
            "validation_mse": validation["mse"],
            "selection_percentage": validation["selection_percentage"],
            "lowest_error_selection_accuracy": validation.get("lowest_error_selection_accuracy"),
            "routing_entropy": validation["routing_entropy"],
            "checkpoint_saved": checkpoint_saved,
            "early_stopping_counter": epochs_without_improvement,
        }
        history.append(row)
        print(
            f"RouterDC epoch {epoch:>3d}/{max_epochs}: "
            f"loss={row['training_loss']:.6f}, expert loss={row['window_expert_loss']:.6f}, "
            f"contrastive={row['window_window_loss']:.6f}, val MAE={row['validation_mae']:.6f}, "
            f"selection acc={row['lowest_error_selection_accuracy']:.3f}, checkpoint saved={checkpoint_saved}"
        )
        if epochs_without_improvement >= patience:
            print(f"RouterDC early stopping after {epoch} epochs")
            break
    selected = _load_torch_checkpoint(checkpoint_path, device)
    router.load_state_dict(selected["router_state_dict"])
    router.eval()
    return tuple(history)


def run_routerdc_hard_input_check_stage(data_dir="datasets/ETTh1", checkpoint_dir="checkpoints", batch_size=512, device="cpu", seed=7):
    _, loaders, scaler, experts, expert_names, _ = _load_router2_environment(data_dir, checkpoint_dir, batch_size, device, seed)
    router = RouterDCHardRouter(num_experts=len(experts)).to(device)
    batch = next(iter(loaders["router_train"]))
    inputs, _, _ = _prepare_forecasting_batch(batch, torch.device(device), scaler)
    similarities, router_probs, selected_expert, prediction, call_counts = routerdc_hard_route_and_predict(router, experts, inputs)
    print("RouterDC hard-router input check")
    print("history:", tuple(inputs.shape))
    print("similarities:", tuple(similarities.shape))
    print("router_probs:", tuple(router_probs.shape))
    print("selected_expert:", tuple(selected_expert.shape))
    print("prediction:", tuple(prediction.shape))
    print("selected expert call counts:", {expert_names[index]: count for index, count in call_counts.items()})
    return router


def run_routerdc_hard_training_stage(
    data_dir="datasets/ETTh1",
    checkpoint_dir="checkpoints",
    batch_size=512,
    max_epochs=50,
    patience=10,
    learning_rate=1e-3,
    device="cpu",
    seed=7,
    error_temperature=0.1,
    router_temperature=1.0,
    use_window_contrastive_loss=True,
    contrastive_lambda=0.1,
    contrastive_temperature=0.2,
    num_clusters=16,
    pca_components=16,
    force_rebuild_cache=False,
):
    _, loaders, scaler, experts, expert_names, _ = _load_router2_environment(data_dir, checkpoint_dir, batch_size, device, seed)
    checkpoint_dir = _resolve_project_path(checkpoint_dir)
    train_cache = build_routerdc_expert_performance_cache(
        experts,
        loaders["router_train"],
        device=device,
        scaler=scaler,
        expert_names=expert_names,
        error_temperature=error_temperature,
        cache_path=routerdc_hard_cache_path(checkpoint_dir, "router_train"),
        force_rebuild=force_rebuild_cache,
    )
    val_cache = build_routerdc_expert_performance_cache(
        experts,
        loaders["router_val"],
        device=device,
        scaler=scaler,
        expert_names=expert_names,
        error_temperature=error_temperature,
        cache_path=routerdc_hard_cache_path(checkpoint_dir, "router_val"),
        force_rebuild=force_rebuild_cache,
    )
    cluster_cache = None
    if use_window_contrastive_loss:
        cluster_cache = build_routerdc_window_cluster_cache(
            train_cache,
            num_clusters=num_clusters,
            pca_components=pca_components,
            seed=seed,
        )
    router = RouterDCHardRouter(
        num_experts=len(experts),
        router_temperature=router_temperature,
    ).to(device)
    optimizer = torch.optim.Adam(router.parameters(), lr=learning_rate)
    return train_routerdc_hard_router(
        router=router,
        experts=experts,
        optimizer=optimizer,
        train_loader=loaders["router_train"],
        val_loader=loaders["router_val"],
        train_cache=train_cache,
        val_cache=val_cache,
        checkpoint_path=routerdc_hard_checkpoint_path(checkpoint_dir, use_window_contrastive_loss),
        max_epochs=max_epochs,
        patience=patience,
        device=device,
        scaler=scaler,
        expert_names=expert_names,
        use_window_contrastive_loss=use_window_contrastive_loss,
        contrastive_lambda=contrastive_lambda,
        contrastive_temperature=contrastive_temperature,
        cluster_cache=cluster_cache,
    )


def run_routerdc_hard_ablation_training_stage(**kwargs):
    histories = {}
    for use_contrastive in (False, True):
        print("\nTraining RouterDC hard router with contrastive loss:", use_contrastive)
        histories[use_contrastive] = run_routerdc_hard_training_stage(
            use_window_contrastive_loss=use_contrastive,
            **kwargs,
        )
    return histories


def run_routerdc_hard_final_test_stage(data_dir="datasets/ETTh1", checkpoint_dir="checkpoints", output_dir="results", batch_size=512, device="cpu", seed=7, error_temperature=0.1, force_rebuild_cache=False):
    _, loaders, scaler, experts, expert_names, _ = _load_router2_environment(data_dir, checkpoint_dir, batch_size, device, seed)
    checkpoint_dir = _resolve_project_path(checkpoint_dir)
    output_dir = _resolve_project_path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    evaluations = {}
    for use_contrastive in (False, True):
        checkpoint_path = routerdc_hard_checkpoint_path(checkpoint_dir, use_contrastive)
        if not checkpoint_path.exists():
            print("Skipping missing RouterDC checkpoint:", checkpoint_path)
            continue
        checkpoint = _load_torch_checkpoint(checkpoint_path, torch.device(device))
        router_config = dict(checkpoint["router_config"])
        router_config.pop("router_type", None)
        router = RouterDCHardRouter(**router_config).to(device)
        router.load_state_dict(checkpoint["router_state_dict"])
        router.eval()
        evaluations[use_contrastive] = evaluate_routerdc_hard_router(
            router,
            experts,
            loaders["test"],
            device=device,
            scaler=scaler,
            expert_names=expert_names,
            performance_cache=None,
            print_shapes=True,
        )

    test_cache = build_routerdc_expert_performance_cache(
        experts,
        loaders["test"],
        device=device,
        scaler=scaler,
        expert_names=expert_names,
        error_temperature=error_temperature,
        cache_path=routerdc_hard_cache_path(checkpoint_dir, "test"),
        force_rebuild=force_rebuild_cache,
    )
    comparison = []
    for expert_index, expert_name in enumerate(expert_names):
        comparison.append({
            "Method": expert_name,
            "Test MAE": test_cache["error_matrix"][:, expert_index].mean().item(),
            "Contrastive": "",
        })
    comparison.append({
        "Method": "Oracle best expert per window",
        "Test MAE": test_cache["error_matrix"].min(dim=1).values.mean().item(),
        "Contrastive": "",
    })
    detailed_results = {
        "selected_expert_names": list(expert_names),
        "individual_expert_mae": {
            expert_name: test_cache["error_matrix"][:, index].mean().item()
            for index, expert_name in enumerate(expert_names)
        },
        "oracle_mae": test_cache["error_matrix"].min(dim=1).values.mean().item(),
        "average_target_suitability_probabilities": {
            expert_name: test_cache["target_probs"][:, index].mean().item()
            for index, expert_name in enumerate(expert_names)
        },
        "routerdc_hard": {},
    }
    for use_contrastive, evaluation in evaluations.items():
        selected = evaluation["selected_expert_indices"]
        best_expert = test_cache["best_expert"]
        label = "with_contrastive" if use_contrastive else "without_contrastive"
        evaluation = dict(evaluation)
        evaluation.update({
            "individual_expert_mae": detailed_results["individual_expert_mae"],
            "oracle_mae": detailed_results["oracle_mae"],
            "lowest_error_selection_accuracy": (selected == best_expert).to(torch.float32).mean().item(),
            "average_target_suitability_probabilities": detailed_results["average_target_suitability_probabilities"],
        })
        evaluation.pop("selected_expert_indices", None)
        detailed_results["routerdc_hard"][label] = evaluation
        comparison.append({
            "Method": f"RouterDC hard ({label})",
            "Test MAE": evaluation["mae"],
            "Contrastive": str(use_contrastive),
        })
    comparison.sort(key=lambda row: row["Test MAE"])
    csv_path = output_dir / "routerdc_hard_test_comparison.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=("Method", "Test MAE", "Contrastive"))
        writer.writeheader()
        writer.writerows(comparison)
    json_path = output_dir / "routerdc_hard_test_metrics.json"
    with json_path.open("w", encoding="utf-8") as file:
        json.dump(detailed_results, file, indent=2)
    print("\nRouterDC hard final comparison")
    for row in comparison:
        print(f"{row['Method']:<40} MAE={row['Test MAE']:.6f}")
    print("Saved:", csv_path)
    print("Saved:", json_path)
    return detailed_results

## Shared Run Settings

In [8]:
DATA_DIR = ROOT / "datasets" / "ETTh1"
CHECKPOINT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
BATCH_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 7
ROUTER_TYPE = "routerdc_hard"
USE_WINDOW_CONTRASTIVE_LOSS = True
ERROR_TEMPERATURE = 0.1
ROUTER_TEMPERATURE = 1.0
CONTRASTIVE_LAMBDA = 0.1
CONTRASTIVE_TEMPERATURE = 0.2
CONTRASTIVE_NUM_CLUSTERS = 16
CONTRASTIVE_PCA_COMPONENTS = 16

AVAILABLE_ROUTER_TYPES = tuple(ROUTER2_SOFT_ROUTER_CLASSES) + ("routerdc_hard",)
if ROUTER_TYPE not in AVAILABLE_ROUTER_TYPES:
    raise ValueError(f"ROUTER_TYPE must be one of {AVAILABLE_ROUTER_TYPES}")

print(f"Device: {DEVICE}")
print(f"Router type: {ROUTER_TYPE}")
if ROUTER_TYPE == "routerdc_hard":
    print(f"RouterDC hard checkpoint: {routerdc_hard_checkpoint_path(CHECKPOINT_DIR, USE_WINDOW_CONTRASTIVE_LOSS)}")
    run_routerdc_hard_dummy_forward_test(num_experts=3, batch_size=4, device=DEVICE)
else:
    print(f"Router 2 checkpoint: {router2_checkpoint_path(CHECKPOINT_DIR, ROUTER_TYPE)}")
    run_router2_dummy_forward_test(router_type=ROUTER_TYPE, num_experts=3, batch_size=4, device=DEVICE)

Device: cuda
Router type: routerdc_hard
RouterDC hard checkpoint: C:\Users\luwil\OneDrive\Documents\Code\BasicTS\checkpoints\best_routerdc_hard_contrastive.pt
RouterDC hard dummy forward-pass test
history: (4, 96, 7)
window_embedding: (4, 64)
similarities: (4, 3)
router_log_probs: (4, 3)
router_probs: (4, 3)
selected_expert: (4,)
probability sum range: 1.0 to 1.0
expert_embeddings grad: True


## Stage A: Check Router 2 Inputs

Set this to `True` after the selected candidate checkpoints exist.

In [9]:
RUN_ROUTER2_INPUT_CHECK = False

if RUN_ROUTER2_INPUT_CHECK and ROUTER_TYPE != "routerdc_hard":
    run_router2_input_check_stage(
        data_dir=DATA_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        seed=SEED,
        router_type=ROUTER_TYPE,
    )
elif ROUTER_TYPE == "routerdc_hard":
    print("ROUTER_TYPE is routerdc_hard; use the RouterDC hard-router input check below.")
else:
    print("Set RUN_ROUTER2_INPUT_CHECK = True to verify Router 2 input/output shapes.")

ROUTER_TYPE is routerdc_hard; use the RouterDC hard-router input check below.


## Stage B: Train Router 2

This trains only `Router2FeatureRouter` and saves `checkpoints/best_router2.pt`.

In [10]:
RUN_ROUTER2_TRAINING = False

if RUN_ROUTER2_TRAINING and ROUTER_TYPE != "routerdc_hard":
    router2_history = run_router2_training_stage(
        data_dir=DATA_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        max_epochs=50,
        patience=10,
        learning_rate=1e-3,
        device=DEVICE,
        seed=SEED,
        router_type=ROUTER_TYPE,
    )
elif ROUTER_TYPE == "routerdc_hard":
    print("ROUTER_TYPE is routerdc_hard; use the RouterDC hard-router training stage below.")
else:
    print("Set RUN_ROUTER2_TRAINING = True to train and save the selected Router 2 checkpoint.")

ROUTER_TYPE is routerdc_hard; use the RouterDC hard-router training stage below.


## Stage C: Final Router 2 Test

This loads `best_router2.pt`, compares it against the expert baselines on the untouched test split, and saves `router2_test_comparison.csv` plus `router2_test_metrics.json`.

In [11]:
RUN_ROUTER2_FINAL_TEST = False

if RUN_ROUTER2_FINAL_TEST and ROUTER_TYPE != "routerdc_hard":
    router2_results = run_router2_test_stage(
        data_dir=DATA_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        output_dir=RESULTS_DIR,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        seed=SEED,
        router_type=ROUTER_TYPE,
    )
elif ROUTER_TYPE == "routerdc_hard":
    print("ROUTER_TYPE is routerdc_hard; use the RouterDC hard-router final test below.")
else:
    print("Set RUN_ROUTER2_FINAL_TEST = True to run the final Router 2 test.")

ROUTER_TYPE is routerdc_hard; use the RouterDC hard-router final test below.


## RouterDC Hard Router Runs

These optional cells are independent from the existing soft `Router2FeatureRouter` stages above.

In [12]:
RUN_ROUTERDC_HARD_INPUT_CHECK = False
RUN_ROUTERDC_HARD_TRAINING = False
RUN_ROUTERDC_HARD_ABLATION_TRAINING = False
RUN_ROUTERDC_HARD_FINAL_TEST = False

if ROUTER_TYPE != "routerdc_hard":
    print("ROUTER_TYPE is not routerdc_hard; these hard-router stages are inactive.")
else:
    if RUN_ROUTERDC_HARD_INPUT_CHECK:
        run_routerdc_hard_input_check_stage(
            data_dir=DATA_DIR,
            checkpoint_dir=CHECKPOINT_DIR,
            batch_size=BATCH_SIZE,
            device=DEVICE,
            seed=SEED,
        )
    else:
        print("Set RUN_ROUTERDC_HARD_INPUT_CHECK = True to verify RouterDC hard-router routing shapes.")

    if RUN_ROUTERDC_HARD_TRAINING:
        routerdc_hard_history = run_routerdc_hard_training_stage(
            data_dir=DATA_DIR,
            checkpoint_dir=CHECKPOINT_DIR,
            batch_size=BATCH_SIZE,
            max_epochs=50,
            patience=10,
            learning_rate=1e-3,
            device=DEVICE,
            seed=SEED,
            error_temperature=ERROR_TEMPERATURE,
            router_temperature=ROUTER_TEMPERATURE,
            use_window_contrastive_loss=USE_WINDOW_CONTRASTIVE_LOSS,
            contrastive_lambda=CONTRASTIVE_LAMBDA,
            contrastive_temperature=CONTRASTIVE_TEMPERATURE,
            num_clusters=CONTRASTIVE_NUM_CLUSTERS,
            pca_components=CONTRASTIVE_PCA_COMPONENTS,
        )
    else:
        print("Set RUN_ROUTERDC_HARD_TRAINING = True to train the selected RouterDC hard checkpoint.")

    if RUN_ROUTERDC_HARD_ABLATION_TRAINING:
        routerdc_hard_ablation_history = run_routerdc_hard_ablation_training_stage(
            data_dir=DATA_DIR,
            checkpoint_dir=CHECKPOINT_DIR,
            batch_size=BATCH_SIZE,
            max_epochs=50,
            patience=10,
            learning_rate=1e-3,
            device=DEVICE,
            seed=SEED,
            error_temperature=ERROR_TEMPERATURE,
            router_temperature=ROUTER_TEMPERATURE,
            contrastive_lambda=CONTRASTIVE_LAMBDA,
            contrastive_temperature=CONTRASTIVE_TEMPERATURE,
            num_clusters=CONTRASTIVE_NUM_CLUSTERS,
            pca_components=CONTRASTIVE_PCA_COMPONENTS,
        )
    else:
        print("Set RUN_ROUTERDC_HARD_ABLATION_TRAINING = True to train with and without contrastive loss.")

    if RUN_ROUTERDC_HARD_FINAL_TEST:
        routerdc_hard_results = run_routerdc_hard_final_test_stage(
            data_dir=DATA_DIR,
            checkpoint_dir=CHECKPOINT_DIR,
            output_dir=RESULTS_DIR,
            batch_size=BATCH_SIZE,
            device=DEVICE,
            seed=SEED,
            error_temperature=ERROR_TEMPERATURE,
        )
    else:
        print("Set RUN_ROUTERDC_HARD_FINAL_TEST = True to run RouterDC hard final inference.")

Set RUN_ROUTERDC_HARD_INPUT_CHECK = True to verify RouterDC hard-router routing shapes.
Set RUN_ROUTERDC_HARD_TRAINING = True to train the selected RouterDC hard checkpoint.
Set RUN_ROUTERDC_HARD_ABLATION_TRAINING = True to train with and without contrastive loss.
Set RUN_ROUTERDC_HARD_FINAL_TEST = True to run RouterDC hard final inference.


## Run All Router 2 Variants

This optional cell runs every router variant defined in this notebook: `original`, `multiscale_tcn_expert_embeddings`, and `routerdc_hard`.

In [13]:
RUN_ALL_ROUTER2_VARIANTS = True
RUN_ALL_SOFT_ROUTERS = True
RUN_ALL_ROUTERDC_HARD = True
RUN_ALL_ROUTERDC_ABLATION = True
RUN_ALL_MAX_EPOCHS = 50
RUN_ALL_PATIENCE = 10

if not RUN_ALL_ROUTER2_VARIANTS:
    print("Set RUN_ALL_ROUTER2_VARIANTS = True to train/test all Router 2 variants.")
else:
    all_router2_variant_results = {}

    if RUN_ALL_SOFT_ROUTERS:
        for router_type in ("original", "multiscale_tcn_expert_embeddings"):
            print("\n" + "=" * 80)
            print(f"Running soft Router 2 variant: {router_type}")
            print("=" * 80)

            run_router2_input_check_stage(
                data_dir=DATA_DIR,
                checkpoint_dir=CHECKPOINT_DIR,
                batch_size=BATCH_SIZE,
                device=DEVICE,
                seed=SEED,
                router_type=router_type,
            )
            history = run_router2_training_stage(
                data_dir=DATA_DIR,
                checkpoint_dir=CHECKPOINT_DIR,
                batch_size=BATCH_SIZE,
                max_epochs=RUN_ALL_MAX_EPOCHS,
                patience=RUN_ALL_PATIENCE,
                learning_rate=1e-3,
                device=DEVICE,
                seed=SEED,
                router_type=router_type,
            )
            results = run_router2_test_stage(
                data_dir=DATA_DIR,
                checkpoint_dir=CHECKPOINT_DIR,
                output_dir=RESULTS_DIR,
                batch_size=BATCH_SIZE,
                device=DEVICE,
                seed=SEED,
                router_type=router_type,
            )
            all_router2_variant_results[router_type] = {
                "history": history,
                "results": results,
            }

    if RUN_ALL_ROUTERDC_HARD:
        print("\n" + "=" * 80)
        print("Running RouterDC hard router")
        print("=" * 80)

        run_routerdc_hard_input_check_stage(
            data_dir=DATA_DIR,
            checkpoint_dir=CHECKPOINT_DIR,
            batch_size=BATCH_SIZE,
            device=DEVICE,
            seed=SEED,
        )
        if RUN_ALL_ROUTERDC_ABLATION:
            hard_history = run_routerdc_hard_ablation_training_stage(
                data_dir=DATA_DIR,
                checkpoint_dir=CHECKPOINT_DIR,
                batch_size=BATCH_SIZE,
                max_epochs=RUN_ALL_MAX_EPOCHS,
                patience=RUN_ALL_PATIENCE,
                learning_rate=1e-3,
                device=DEVICE,
                seed=SEED,
                error_temperature=ERROR_TEMPERATURE,
                router_temperature=ROUTER_TEMPERATURE,
                contrastive_lambda=CONTRASTIVE_LAMBDA,
                contrastive_temperature=CONTRASTIVE_TEMPERATURE,
                num_clusters=CONTRASTIVE_NUM_CLUSTERS,
                pca_components=CONTRASTIVE_PCA_COMPONENTS,
            )
        else:
            hard_history = run_routerdc_hard_training_stage(
                data_dir=DATA_DIR,
                checkpoint_dir=CHECKPOINT_DIR,
                batch_size=BATCH_SIZE,
                max_epochs=RUN_ALL_MAX_EPOCHS,
                patience=RUN_ALL_PATIENCE,
                learning_rate=1e-3,
                device=DEVICE,
                seed=SEED,
                error_temperature=ERROR_TEMPERATURE,
                router_temperature=ROUTER_TEMPERATURE,
                use_window_contrastive_loss=USE_WINDOW_CONTRASTIVE_LOSS,
                contrastive_lambda=CONTRASTIVE_LAMBDA,
                contrastive_temperature=CONTRASTIVE_TEMPERATURE,
                num_clusters=CONTRASTIVE_NUM_CLUSTERS,
                pca_components=CONTRASTIVE_PCA_COMPONENTS,
            )
        hard_results = run_routerdc_hard_final_test_stage(
            data_dir=DATA_DIR,
            checkpoint_dir=CHECKPOINT_DIR,
            output_dir=RESULTS_DIR,
            batch_size=BATCH_SIZE,
            device=DEVICE,
            seed=SEED,
            error_temperature=ERROR_TEMPERATURE,
        )
        all_router2_variant_results["routerdc_hard"] = {
            "history": hard_history,
            "results": hard_results,
        }

    print("\nFinished all requested Router 2 variants.")
    print("Soft-router results are saved as results/router2_*_test_comparison.csv")
    print("RouterDC hard results are saved as results/routerdc_hard_test_comparison.csv")


Running soft Router 2 variant: original

Chronological split audit (end index is exclusive)
split              start       end   timestamps   valid windows
expert_train           0      7200         7200            7093
expert_val          7200      8640         1440            1333
router_train        8640     10800         2160            2053
router_val         10800     11520          720             613
test               11520     14400         2880            2773

Example batch shapes (the expert-train scaler is reused)
expert_train   Sample input: [96, 7]  Sample target: [12, 7]  Input: [512, 96, 7]  Target: [512, 12, 7]
expert_val     Sample input: [96, 7]  Sample target: [12, 7]  Input: [512, 96, 7]  Target: [512, 12, 7]
router_train   Sample input: [96, 7]  Sample target: [12, 7]  Input: [512, 96, 7]  Target: [512, 12, 7]
router_val     Sample input: [96, 7]  Sample target: [12, 7]  Input: [512, 96, 7]  Target: [512, 12, 7]
test           Sample input: [96, 7]  Sample targ